# PINN Explainability — Interactive 3D Visualizations (napari)

Companion to `pinns-strain-sota-adaptive-2.ipynb`. Trains (or loads) the PINN at **six sampling
fractions** — 1%, 5%, 10%, 25%, 50%, 75%, matching the main notebook — with the same fixed
pipeline (ω₀ = 30, frozen-EMA physics normalisation, λ ramp) and opens five interactive
**napari** windows. Rotate with the mouse, toggle layers with the eye icons, and use the sliders
below the canvas (hover a slider and press ▶, or right-click it, to play it as an animation).

| View | Sliders | What it explains |
|---|---|---|
| **A. Layer stack** | sampling %, statistic | How spatial features form layer by layer, in the *final trained* network: each sine layer's activations (PCA-1/2/3 or mean magnitude) as a translucent plane in a 3D stack — and how the internal representation changes from 1% to 75% sampling. |
| **B. Field explorer** | — (layer toggles) | Every component — ε_xx, ε_yy, ε_xy and θ (rotation) — as 3D surfaces: ground truth, PINN prediction, and \|error\| per field, plus the equilibrium-residual map and the training points. Set `VIZ_FRAC` in the cell to explore another fraction. |
| **C. Sine gratings** | sampling %, neuron | The network's "basis": first-layer neurons are plane-wave gratings sin(ω₀(w·x)+b); slide through all 128 neurons (variance-sorted) and compare the basis learned at each fraction. |
| **D. Training evolution** | sampling %, field, snapshot | Watch each field emerge during optimisation: ~26 log-spaced epoch snapshots per fraction, with a ground-truth layer for comparison. |
| **E. Layer stack, training evolution** | sampling %, snapshot | The *dynamic* counterpart to View A: the same layer stack (measured field, five sine layers, reconstruction), plus the elastic-equilibrium and Saint-Venant compatibility residuals stacked on top, all animated across the same ~26 epochs as View D. Watch the layers organise into structure while the physics-residual planes darken — i.e. watch the network learn the field *and* satisfy the physics prior at the same time. A text overlay reports λ_phys(t) and the mean residuals numerically. |

**First run trains all six fractions** (a few minutes on Apple Silicon, thanks to early
stopping); afterwards each fraction is cached in `outputs/viz3d/` as `model_{pct}pct.pt` +
`snapshots_{pct}pct.npz` (the latter now also holds per-epoch layer activity and physics
residuals for View E, so caches are a few times larger than a plain output-only cache).
Set `RETRAIN = True` to force retraining. Requires `napari` with a Qt backend
(`pip install napari pyqt6` — already in the `pinns` env). Run cells in order — the view cells
share arrays defined in the earlier ones.


In [1]:
# ---- configuration -------------------------------------------------------
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
import matplotlib

try:
    import napari
    from napari.utils import Colormap
    HAS_NAPARI = True
except Exception:
    HAS_NAPARI = False
    napari = None
    Colormap = None
    print('napari not available — napari view cells (A–E) will be skipped.\n'
          'Run the ipywidgets + plotly cells at the bottom of this notebook instead.')

from sklearn.decomposition import PCA

SEED    = 42
FRACS   = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75]   # sampling fractions (as in the main notebook)
DS      = 2            # downsample factor for the 3D views (2 -> 90x200 grids)
RETRAIN = False
OMEGA_0 = 30.0
HIDDEN, LAYERS = 128, 6
EPOCHS  = 3000
SNAP_EPOCHS = sorted({0, EPOCHS, *np.geomspace(1, EPOCHS, 25).round().astype(int)})
N_SNAP  = len(SNAP_EPOCHS) + 1                    # +1: final best-restored state

FIELDS       = ['e_xx', 'e_yy', 'e_xy', 'theta']
FIELD_LABELS = ['ε_xx', 'ε_yy', 'ε_xy', 'θ (rotation)']
F0 = FRACS.index(0.10) if 0.10 in FRACS else 0    # default position of the sampling slider

device  = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')
out_dir = Path('outputs/viz3d'); out_dir.mkdir(parents=True, exist_ok=True)

def mpl_cmap(name):
    return Colormap(matplotlib.colormaps[name](np.linspace(0, 1, 256)), name=name)

if HAS_NAPARI:
    CM_DIV, CM_HOT, CM_SEQ = mpl_cmap('RdBu'), mpl_cmap('hot'), mpl_cmap('viridis')
else:
    CM_DIV = CM_HOT = CM_SEQ = None

print(f'device={device}  napari={HAS_NAPARI}  cache -> {out_dir.resolve()}')
print(f'{len(FRACS)} fractions; {len(SNAP_EPOCHS)} snapshot epochs: {list(map(int, SNAP_EPOCHS))}')

device=mps  napari=True  cache -> /Users/robertoreis/Documents/codes/pinns-4dstem/outputs/viz3d
6 fractions; 25 snapshot epochs: [0, 1, 2, 3, 4, 5, 7, 10, 14, 20, 28, 39, 55, 76, 107, 149, 208, 290, 405, 566, 790, 1103, 1539, 2149, 3000]


In [2]:
# ---- data (identical preprocessing to the main notebook) -----------------
root = Path('data')
e_xx = np.load(root/'strain_exx.npy'); e_yy = np.load(root/'strain_eyy.npy'); e_xy = np.load(root/'strain_exy.npy')
theta = np.arctan2(e_xy, (e_xx - e_yy)/2.0 + 1e-10)
H, W = e_xx.shape
Hd, Wd = H // DS, W // DS
raw = {'e_xx': e_xx, 'e_yy': e_yy, 'e_xy': e_xy, 'theta': theta}
gt_maps = np.stack([raw[n] for n in FIELDS])       # (4, H, W) measured fields
scalers = {n: {'mean': float(a.mean()), 'std': float(a.std()) + 1e-8} for n, a in raw.items()}

x_lin = torch.linspace(0, 1, W); y_lin = torch.linspace(0, 1, H)
X, Y = torch.meshgrid(x_lin, y_lin, indexing='xy')
coords = torch.stack([X.flatten(), Y.flatten()], dim=1).to(device)
coords_ds = coords.reshape(H, W, 2)[::DS, ::DS].reshape(-1, 2).contiguous()  # grid for View E captures
tgt = torch.stack([torch.tensor(((raw[n] - scalers[n]['mean'])/scalers[n]['std']).flatten(), dtype=torch.float32)
                   for n in FIELDS], dim=1).to(device)

sel_by_frac = {f: np.random.RandomState(SEED + int(f*1000))
                    .choice(np.arange(H*W), int(np.ceil(f*H*W)), replace=False) for f in FRACS}
print(f'{H}x{W} grid; training points per fraction:',
      {f'{f*100:g}%': len(s) for f, s in sel_by_frac.items()})

180x400 grid; training points per fraction: {'1%': 720, '5%': 3600, '10%': 7200, '25%': 18000, '50%': 36000, '75%': 54000}


In [3]:
# ---- model + per-fraction training with snapshots (fixed pipeline) --------
class SirenLayer(nn.Module):
    def __init__(self, i, o, first=False, w0=OMEGA_0):
        super().__init__(); self.w0 = w0; self.linear = nn.Linear(i, o)
        with torch.no_grad():
            self.linear.weight.uniform_(-1/i, 1/i) if first else \
                self.linear.weight.uniform_(-np.sqrt(6/i)/w0, np.sqrt(6/i)/w0)
    def forward(self, x): return torch.sin(self.w0 * self.linear(x))

class SirenNet(nn.Module):
    def __init__(self, hidden=HIDDEN, layers=LAYERS, w0=OMEGA_0):
        super().__init__()
        self.first = SirenLayer(2, hidden, True, w0)
        self.hidden_layers = nn.ModuleList([SirenLayer(hidden, hidden, False, w0) for _ in range(layers-2)])
        self.skip_proj = nn.Linear(hidden, hidden)
        self.final = nn.Linear(hidden, 4)
        with torch.no_grad(): self.final.weight.uniform_(-np.sqrt(6/hidden), np.sqrt(6/hidden))
    def forward(self, x):
        h = self.first(x)
        for i, l in enumerate(self.hidden_layers):
            h = l(h) + self.skip_proj(h) if (i % 2 == 0 and i > 0) else l(h)
        return self.final(h)

def physics_residuals(model, n=2000, nu=0.27):
    pts = torch.rand(n, 2, device=device).requires_grad_(True)
    out = model(pts); exx, eyy, exy = out[:, 0], out[:, 1], out[:, 2]
    g = lambda o: torch.autograd.grad(o, pts, torch.ones_like(o), create_graph=True, retain_graph=True)[0]
    gx, gy, gxy = g(exx), g(eyy), g(exy)
    c = (1 - nu)/2
    L_eq = ((gx[:, 0] + nu*gy[:, 0] + c*gxy[:, 1])**2 + (c*gxy[:, 0] + gy[:, 1] + nu*gx[:, 1])**2).mean()
    L_co = (g(gx[:, 1])[:, 1] + g(gy[:, 0])[:, 0] - 2*g(gxy[:, 0])[:, 1]).pow(2).mean()
    return L_eq, L_co

def to_map(pred_col, name='e_xx'):
    return (pred_col*scalers[name]['std'] + scalers[name]['mean']).reshape(H, W)

def hidden_layer_activity(model, pts):
    """Mean |activation| of every sine layer at fixed grid points -> (n_layers, Hd, Wd).
    Used to watch each layer's engagement sharpen over training in View E."""
    hooks, store = [], []
    def _hook(m, i, o): store.append(o.detach())
    for lyr in [model.first, *model.hidden_layers]:
        hooks.append(lyr.register_forward_hook(_hook))
    with torch.no_grad():
        store.clear(); model(pts)
    for h in hooks: h.remove()
    return np.stack([np.abs(s.cpu().numpy()).mean(1).reshape(Hd, Wd) for s in store])

def physics_residual_maps(model, pts, nu=0.27):
    """Elementwise (unreduced) equilibrium & compatibility residuals on a fixed grid,
    the same quantities trained on in physics_residuals() above but on a deterministic
    grid instead of random collocation points, for visualising in View E."""
    p = pts.clone().requires_grad_(True)
    out = model(p); exx, eyy, exy = out[:, 0], out[:, 1], out[:, 2]
    g = lambda o: torch.autograd.grad(o, p, torch.ones_like(o), create_graph=True, retain_graph=True)[0]
    gx, gy, gxy = g(exx), g(eyy), g(exy)
    c = (1 - nu)/2
    eq = (gx[:, 0] + nu*gy[:, 0] + c*gxy[:, 1])**2 + (c*gxy[:, 0] + gy[:, 1] + nu*gx[:, 1])**2
    co = (g(gx[:, 1])[:, 1] + g(gy[:, 0])[:, 0] - 2*g(gxy[:, 0])[:, 1])**2
    return (eq.detach().cpu().numpy().reshape(Hd, Wd),
            co.detach().cpu().numpy().reshape(Hd, Wd))

def train_frac(frac):
    tag = f'{frac*100:g}pct'
    ckpt, snp = out_dir/f'model_{tag}.pt', out_dir/f'snapshots_{tag}.npz'
    if ckpt.exists() and snp.exists() and not RETRAIN:
        model = SirenNet().to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
        print(f'  {tag}: loaded cached checkpoint + snapshots')
        return model, dict(np.load(snp))
    torch.manual_seed(SEED)
    model = SirenNet().to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
    sched = torch.optim.lr_scheduler.StepLR(opt, 500, 0.95)
    idx = sel_by_frac[frac].copy(); np.random.RandomState(SEED).shuffle(idx)
    n_val = int(0.10*len(idx))
    val_i = torch.as_tensor(idx[:n_val], dtype=torch.long, device=device)
    tr_i  = torch.as_tensor(idx[n_val:], dtype=torch.long, device=device)
    tr_in, tr_tgt, val_in, val_tgt = coords[tr_i], tgt[tr_i], coords[val_i], tgt[val_i]
    ema, best, pc, best_state = {}, float('inf'), 0, None
    snap_pred, snap_ep, snap_val = [], [], []
    snap_layers, snap_eq, snap_co = [], [], []
    def snapshot(ep, vl):
        model.eval()
        with torch.no_grad():
            p = model(coords).cpu().numpy()
        maps = np.stack([to_map(p[:, i], n)[::DS, ::DS] for i, n in enumerate(FIELDS)])
        snap_pred.append(maps.astype(np.float32)); snap_ep.append(ep); snap_val.append(vl)
        snap_layers.append(hidden_layer_activity(model, coords_ds).astype(np.float32))
        eq_map, co_map = physics_residual_maps(model, coords_ds)
        snap_eq.append(eq_map.astype(np.float32)); snap_co.append(co_map.astype(np.float32))
    for ep in range(EPOCHS + 1):
        model.train(); opt.zero_grad()
        ld = ((model(tr_in) - tr_tgt)**2).mean()
        lam = 1.0 - np.exp(-ep/500.0)
        L_eq, L_co = physics_residuals(model)
        ema['n'] = ema.get('n', 0) + 1
        if ema['n'] <= 200:
            for k, L in (('eq', L_eq), ('co', L_co)):
                v = float(L.detach()); ema[k] = v if k not in ema else 0.99*ema[k] + 0.01*v
        lp = 0.1*L_eq/(ema['eq'] + 1e-12) + 0.1*L_co/(ema['co'] + 1e-12)
        (ld + lam*lp).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        with torch.no_grad():
            model.eval(); vl = ((model(val_in) - val_tgt)**2).mean().item()
        if ep in SNAP_EPOCHS: snapshot(ep, vl)
        if best - vl > 0: best, pc, best_state = vl, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}
        else: pc += 1
        if pc >= 300:
            print(f'  {tag}: early stop @ {ep}'); break
    if best_state: model.load_state_dict(best_state)
    snapshot(ep, best)                        # final (best-restored) state
    while len(snap_pred) < N_SNAP:            # pad early-stopped runs to a fixed length
        snap_pred.append(snap_pred[-1]); snap_ep.append(snap_ep[-1]); snap_val.append(snap_val[-1])
        snap_layers.append(snap_layers[-1]); snap_eq.append(snap_eq[-1]); snap_co.append(snap_co[-1])
    snaps = {'pred': np.stack(snap_pred), 'epoch': np.array(snap_ep), 'val': np.array(snap_val),
             'layers': np.stack(snap_layers), 'eq': np.stack(snap_eq), 'co': np.stack(snap_co)}
    torch.save(model.state_dict(), ckpt); np.savez(snp, **snaps)
    print(f'  {tag}: trained (best val {best:.4f})')
    return model, snaps

models, snaps_by_frac, preds = {}, {}, {}
for frac in FRACS:
    models[frac], snaps_by_frac[frac] = train_frac(frac)
    with torch.no_grad():
        models[frac].eval(); p = models[frac](coords).cpu().numpy()
    preds[frac] = np.stack([to_map(p[:, i], n) for i, n in enumerate(FIELDS)])   # (4, H, W)

print('\nR²        ' + ''.join(f'{l:>14s}' for l in FIELD_LABELS))
for frac in FRACS:
    r2 = [1 - np.sum((gt_maps[c] - preds[frac][c])**2)/np.sum((gt_maps[c] - gt_maps[c].mean())**2)
          for c in range(len(FIELDS))]
    print(f'{frac*100:>7.0f}%  ' + ''.join(f'{v:14.3f}' for v in r2))

  1pct: loaded cached checkpoint + snapshots
  5pct: loaded cached checkpoint + snapshots
  10pct: loaded cached checkpoint + snapshots
  25pct: loaded cached checkpoint + snapshots
  50pct: loaded cached checkpoint + snapshots
  75pct: loaded cached checkpoint + snapshots

R²                  ε_xx          ε_yy          ε_xy  θ (rotation)
      1%           0.428         0.635         0.542         0.289
      5%           0.729         0.825         0.778         0.517
     10%           0.807         0.861         0.839         0.619
     25%           0.859         0.915         0.893         0.722
     50%           0.863         0.904         0.889         0.753
     75%           0.862         0.890         0.887         0.748


In [4]:
# ---- layer activations + PCA stats, per fraction ---------------------------
def stats_for(a):
    pca = PCA(n_components=3).fit(a[::7])          # fit on a subsample for speed
    p = pca.transform(a)
    ds = lambda m: m.reshape(H, W)[::DS, ::DS]
    return {'PCA-1': ds(p[:, 0]), 'PCA-2': ds(p[:, 1]), 'PCA-3': ds(p[:, 2]),
            'mean |activation|': ds(np.abs(a).mean(1))}
STAT_KEYS = ['PCA-1', 'PCA-2', 'PCA-3', 'mean |activation|']

def layer_stats_for(model):
    hooks, store = [], []
    def _hook(m, i, o): store.append(o.detach())
    for lyr in [model.first, *model.hidden_layers]:
        hooks.append(lyr.register_forward_hook(_hook))
    with torch.no_grad():
        store.clear(); model(coords)
    for h in hooks: h.remove()
    return [stats_for(s.cpu().numpy()) for s in store]

stats_by_frac = {f: layer_stats_for(models[f]) for f in FRACS}
n_layers = len(stats_by_frac[FRACS[0]])
layer_names = ['sine layer 1 (input)'] + [f'sine layer {i+2}' for i in range(n_layers-1)]
print(f'PCA stats for {n_layers} layers x {len(FRACS)} fractions')

PCA stats for 5 layers x 6 fractions


In [5]:
# ---- View A: 3D layer stack (sliders: sampling %, statistic) ---------------
# Planes bottom->top: measured ε_xx, the five sine layers, reconstructed ε_xx.
# The trunk is shared by all four outputs, so the stack is anchored on ε_xx;
# views B/D cover the other components. Slide 'sampling %' to see how the
# internal representation sharpens from 1% to 75%. This is the FINAL trained
# state only — View E shows the same stack animated across training.
def norm_sym(a, p=99.5):                  # robust map to [-1, 1], zero preserved
    return np.clip(a / (np.percentile(np.abs(a), p) + 1e-12), -1, 1)
def norm_minmax(a):                       # map [min, max] -> [-1, 1]
    return (a - a.min()) / (a.max() - a.min() + 1e-12) * 2 - 1

n_f, n_s = len(FRACS), len(STAT_KEYS)
gt_ds   = gt_maps[:, ::DS, ::DS]                                  # (4, Hd, Wd)
pred_ds = np.stack([preds[f][:, ::DS, ::DS] for f in FRACS])      # (n_f, 4, Hd, Wd)
SPACING = 60                              # z distance between planes (px)

def stat_stack(st):                       # (n_stats, Hd, Wd) in display units
    return np.stack([norm_minmax(st[k]) if k == 'mean |activation|' else norm_sym(st[k])
                     for k in STAT_KEYS])

planes  = [('input: measured ε_xx (bottom)',
            np.broadcast_to(np.stack([norm_sym(gt_ds[0])]*n_s), (n_f, n_s, Hd, Wd)).copy())]
planes += [(layer_names[k], np.stack([stat_stack(stats_by_frac[f][k]) for f in FRACS]))
           for k in range(n_layers)]
planes += [('output: reconstructed ε_xx (top)',
            np.stack([np.stack([norm_sym(pred_ds[i, 0])]*n_s) for i in range(n_f)]))]

viewerA = napari.Viewer(ndisplay=3, title='A — what each SIREN layer represents')
for k, (nm, arr) in enumerate(planes):    # arr[:, :, None]: (frac, stat, z=1, y, x)
    viewerA.add_image(arr[:, :, None], name=nm, colormap=CM_DIV, contrast_limits=(-1, 1),
                      opacity=0.9, blending='translucent',
                      scale=(1, 1, 1, DS, DS), translate=(0, 0, k*SPACING, 0, 0))
viewerA.dims.axis_labels = ('sampling %', 'stat', 'z', 'y', 'x')
viewerA.dims.set_current_step(0, F0)      # start at 10% sampling, PCA-1
viewerA.dims.set_current_step(1, 0)
viewerA.reset_view()
viewerA.camera.angles = (-15, 30, 65)     # starting viewpoint — drag to rotate
viewerA.text_overlay.visible = True
def _labelA(event=None):
    fi, si = viewerA.dims.current_step[0], viewerA.dims.current_step[1]
    viewerA.text_overlay.text = f'sampling = {FRACS[fi]*100:g}%    colour = {STAT_KEYS[si]}'
viewerA.dims.events.current_step.connect(_labelA); _labelA()

In [6]:
# ---- View B: field explorer (all components, one sampling fraction) --------
# One Surface per field x {ground truth, PINN, |error|} plus the equilibrium
# residual — toggle with the eye icons. Height and colour = field value.
# Change VIZ_FRAC and rerun the cell to open a window for another fraction.
VIZ_FRAC = FRACS[F0]                      # set to any value in FRACS (e.g. 0.01) and rerun
p4 = preds[VIZ_FRAC]                      # (4, H, W)

px_um = 0.017872741
E_GPa, nu = 150.0, 0.27
fpl = E_GPa/(1 - nu**2); G = E_GPa/(2*(1 + nu))
sxx, syy, sxy = fpl*(p4[0] + nu*p4[1]), fpl*(p4[1] + nu*p4[0]), G*p4[2]
d = lambda a: np.gradient(a, px_um)
dsxx_dy, dsxx_dx = d(sxx); dsyy_dy, dsyy_dx = d(syy); dsxy_dy, dsxy_dx = d(sxy)
equil_map = np.sqrt((dsxx_dx + dsxy_dy)**2 + (dsxy_dx + dsyy_dy)**2)

Z_SCALE = 60.0
def grid_surface(field, zlim=None):       # (vertices, faces, values) for napari
    hh, ww = field.shape
    lo, hi = zlim if zlim else (float(field.min()), float(field.max()))
    zz = (field - lo)/(hi - lo + 1e-12)*Z_SCALE
    yy, xx = np.mgrid[0:hh, 0:ww].astype(float)*DS
    verts = np.column_stack([zz.ravel(), yy.ravel(), xx.ravel()])
    idx = np.arange(hh*ww).reshape(hh, ww)
    faces = np.vstack([np.column_stack([idx[:-1, :-1].ravel(), idx[1:, :-1].ravel(), idx[:-1, 1:].ravel()]),
                       np.column_stack([idx[1:, :-1].ravel(), idx[1:, 1:].ravel(), idx[:-1, 1:].ravel()])])
    return verts, faces, field.ravel().astype(np.float32)

viewerB = napari.Viewer(ndisplay=3, title=f'B — field explorer ({VIZ_FRAC*100:g}% sampling)')
for c, lbl in enumerate(FIELD_LABELS):
    gt_c, pr_c = gt_ds[c], p4[c][::DS, ::DS]
    err_c = np.abs(pr_c - gt_c)
    zl = (float(min(gt_c.min(), pr_c.min())), float(max(gt_c.max(), pr_c.max())))
    vl = max(abs(zl[0]), abs(zl[1]))
    viewerB.add_surface(grid_surface(gt_c, zl), name=f'{lbl} · ground truth', colormap=CM_DIV,
                        contrast_limits=(-vl, vl), visible=False, shading='smooth')
    viewerB.add_surface(grid_surface(pr_c, zl), name=f'{lbl} · PINN', colormap=CM_DIV,
                        contrast_limits=(-vl, vl), visible=(c == 0), shading='smooth')
    viewerB.add_surface(grid_surface(err_c), name=f'{lbl} · |error|', colormap=CM_HOT,
                        contrast_limits=(0, float(err_c.max()) + 1e-12), visible=False, shading='smooth')
res_ds = equil_map[::DS, ::DS]
viewerB.add_surface(grid_surface(res_ds), name='equilibrium residual [GPa/µm]', colormap=CM_HOT,
                    contrast_limits=(0, float(res_ds.max())), visible=False, shading='smooth')
ys, xs = np.unravel_index(sel_by_frac[VIZ_FRAC], (H, W))
zl0 = (float(min(gt_ds[0].min(), p4[0].min())), float(max(gt_ds[0].max(), p4[0].max())))
zpts = (e_xx[ys, xs] - zl0[0])/(zl0[1] - zl0[0] + 1e-12)*Z_SCALE + 1.0
viewerB.add_points(np.column_stack([zpts, ys, xs]), size=3, face_color='black', visible=False,
                   name=f'training points ({VIZ_FRAC*100:g}%, ε_xx height)')
viewerB.dims.axis_labels = ('z', 'y', 'x')
viewerB.reset_view()
viewerB.camera.angles = (-15, 30, 65)

In [7]:
# ---- View C: first-layer sine gratings (sliders: sampling %, neuron) -------
# The network's basis: every first-layer neuron is a plane-wave grating
# sin(ω₀(w·x)+b). Slide through all 128 neurons (sorted by activation
# variance) and across fractions; press ▶ on the neuron slider to cycle.
grat, orders, w1s = [], [], []
for fr in FRACS:
    with torch.no_grad():
        a1 = torch.sin(models[fr].first.w0 * models[fr].first.linear(coords)).cpu().numpy()
    o = np.argsort(-a1.var(0))            # most active neurons first
    grat.append(a1.T[o].reshape(-1, H, W)[:, ::DS, ::DS])
    orders.append(o); w1s.append(models[fr].first.linear.weight.detach().cpu().numpy())
gratings = np.stack(grat).astype(np.float32)      # (n_frac, 128, Hd, Wd)

viewerC = napari.Viewer(title='C — the learned basis: first-layer gratings')
viewerC.add_image(gratings, name='sin(ω₀(w·x)+b)', colormap=CM_DIV,
                  contrast_limits=(-1, 1), scale=(1, 1, DS, DS))
viewerC.dims.axis_labels = ('sampling %', 'neuron (variance rank)', 'y', 'x')
viewerC.dims.set_current_step(0, F0)
viewerC.dims.set_current_step(1, 0)
viewerC.text_overlay.visible = True
def _labelC(event=None):
    fi, ri = viewerC.dims.current_step[0], viewerC.dims.current_step[1]
    n = int(orders[fi][ri])
    fx, fy = OMEGA_0*w1s[fi][n]/(2*np.pi)
    viewerC.text_overlay.text = (f'{FRACS[fi]*100:g}% sampling   neuron {n} '
                                 f'(rank {ri+1}/{gratings.shape[1]})   '
                                 f'spatial frequency ≈ ({fx:+.1f}, {fy:+.1f}) cycles/unit')
viewerC.dims.events.current_step.connect(_labelC); _labelC()

In [8]:
# ---- View D: training evolution (sliders: sampling %, field, snapshot) -----
# ~26 log-spaced epoch snapshots per fraction, for all four fields. Each field
# is normalised by a robust scale of its ground truth so strains and rotation
# share one contrast range; press ▶ on the 'snapshot' slider to play. Toggle
# 'ground truth' in the layer list to compare (it follows the same sliders).
scale_c = np.array([np.percentile(np.abs(gt_ds[c]), 99.5) + 1e-12 for c in range(len(FIELDS))])
pred_stack = np.stack([snaps_by_frac[fr]['pred'] for fr in FRACS])   # (n_f, n_snap, 4, Hd, Wd)
disp = (pred_stack.transpose(0, 2, 1, 3, 4) / scale_c[None, :, None, None, None]).astype(np.float32)
gt_disp = np.broadcast_to((gt_ds/scale_c[:, None, None])[None, :, None], disp.shape).astype(np.float32)

viewerD = napari.Viewer(title='D — training evolution')
viewerD.add_image(gt_disp, name='ground truth', colormap=CM_DIV,
                  contrast_limits=(-1, 1), scale=(1, 1, 1, DS, DS), visible=False)
viewerD.add_image(disp, name='PINN prediction', colormap=CM_DIV,
                  contrast_limits=(-1, 1), scale=(1, 1, 1, DS, DS))
viewerD.dims.axis_labels = ('sampling %', 'field', 'snapshot', 'y', 'x')
viewerD.dims.set_current_step(0, F0)
viewerD.dims.set_current_step(1, 0)
viewerD.dims.set_current_step(2, 0)
viewerD.text_overlay.visible = True
def _labelD(event=None):
    fi, ci, ki = viewerD.dims.current_step[:3]
    ep = int(snaps_by_frac[FRACS[fi]]['epoch'][ki]); vl = snaps_by_frac[FRACS[fi]]['val'][ki]
    viewerD.text_overlay.text = (f'{FRACS[fi]*100:g}% sampling   {FIELD_LABELS[ci]}   '
                                 f'epoch {ep}   val loss {vl:.3f}')
viewerD.dims.events.current_step.connect(_labelD); _labelD()

In [9]:
# ---- View E: layer stack, training evolution (sliders: sampling %, snapshot)
# The dynamic counterpart to View A: the same bottom-to-top stack (measured
# ε_xx, the five sine layers, reconstructed ε_xx), plus the elastic-
# equilibrium and Saint-Venant compatibility residuals stacked on top — all
# captured at the same ~26 log-spaced epochs used in View D. Slide 'snapshot'
# (or press ▶ on it) to watch the layers organise into structure while the
# two residual planes darken, i.e. watch the network learn the field *and*
# incorporate the physics prior at the same time. A text overlay reports the
# physics-weight ramp λ_phys(t) and the mean residuals numerically.
#
# Each plane is colour-scaled by ONE fixed range spanning all epochs of its
# fraction (not renormalised every frame), so the growth in layer activity
# and the shrinkage of the residuals stay visible across the snapshot slider
# — unlike View A, which only shows the final trained state.
def fixed_range_stack(arr, sym=False, logscale=False, p=99.5):
    """arr: (n_frac, n_snap, Hd, Wd) -> same shape, each fraction rescaled by
    one fixed range spanning all its epochs (so temporal change stays visible)."""
    out = np.empty_like(arr, dtype=np.float32)
    for i in range(arr.shape[0]):
        a = np.log10(arr[i] + 1e-8) if logscale else arr[i]
        if sym:
            lim = np.percentile(np.abs(a), p) + 1e-12
            out[i] = np.clip(a / lim, -1, 1)
        else:
            lo, hi = np.percentile(a, 100 - p), np.percentile(a, p)
            out[i] = np.clip((a - lo) / (hi - lo + 1e-12), 0, 1)
    return out

gt_bcast   = np.broadcast_to(gt_ds[0][None, None], (len(FRACS), N_SNAP, Hd, Wd))
# backward-compatible snapshots: old caches may use 'units' instead of 'layers'
layer_key = 'layers' if 'layers' in snaps_by_frac[FRACS[0]] else 'units'
layers_raw = np.stack([snaps_by_frac[fr][layer_key] for fr in FRACS])

# expected downstream: (n_f, n_snap, n_layers, Hd, Wd)
if layers_raw.ndim == 6:  # e.g. (n_f, n_snap, n_layers, n_units, Hd, Wd)
    layers_raw = np.abs(layers_raw).mean(axis=3)
elif layers_raw.ndim != 5:
    raise ValueError(f"Unexpected '{layer_key}' shape: {layers_raw.shape}")
out_raw    = np.stack([snaps_by_frac[fr]['pred'][:, 0] for fr in FRACS])   # ε_xx (n_f, n_snap, Hd, Wd)
eq_raw     = np.stack([snaps_by_frac[fr]['eq'] for fr in FRACS])
co_raw     = np.stack([snaps_by_frac[fr]['co'] for fr in FRACS])

measured_stack = fixed_range_stack(gt_bcast, sym=True)
layer_stacks   = [fixed_range_stack(layers_raw[:, :, l], sym=False) for l in range(layers_raw.shape[2])]
output_stack   = fixed_range_stack(out_raw, sym=True)
eq_stack       = fixed_range_stack(eq_raw, sym=False, logscale=True)
co_stack       = fixed_range_stack(co_raw, sym=False, logscale=True)

planes_e  = [('measured ε_xx (bottom, reference — constant)', measured_stack, CM_DIV, (-1, 1))]
planes_e += [(f'{layer_names[l]} — mean |activation|', layer_stacks[l], CM_SEQ, (0, 1))
             for l in range(len(layer_stacks))]
planes_e += [('reconstructed ε_xx', output_stack, CM_DIV, (-1, 1))]
planes_e += [('elastic-equilibrium residual (log scale)', eq_stack, CM_HOT, (0, 1))]
planes_e += [('Saint-Venant compatibility residual (log scale)', co_stack, CM_HOT, (0, 1))]

# equalise x/y/z extents so the stack renders as a cube, not a stretched slab:
# natural extents are x=Wd*DS, y=Hd*DS, z=(n_planes-1)*spacing -- pick the largest
# (x) as the common cube side and scale y and the z-spacing to match it.
CUBE_SIDE = Wd * DS
Y_SCALE   = CUBE_SIDE / Hd
SPACING_E = CUBE_SIDE / (len(planes_e) - 1)
viewerE = napari.Viewer(ndisplay=3, title='E — layer stack, training evolution')
for k, (nm, arr, cmap, clim) in enumerate(planes_e):
    viewerE.add_image(arr[:, :, None], name=nm, colormap=cmap, contrast_limits=clim,
                      opacity=0.9, blending='translucent',
                      scale=(1, 1, 1, Y_SCALE, DS), translate=(0, 0, k*SPACING_E, 0, 0))
viewerE.dims.axis_labels = ('sampling %', 'snapshot', 'z', 'y', 'x')
viewerE.dims.set_current_step(0, F0)
viewerE.dims.set_current_step(1, 0)
viewerE.reset_view()
viewerE.camera.angles = (-15, 30, 65)
viewerE.text_overlay.visible = True
def _labelE(event=None):
    fi, ki = viewerE.dims.current_step[0], viewerE.dims.current_step[1]
    fr = FRACS[fi]; sf = snaps_by_frac[fr]
    ep = int(sf['epoch'][ki]); lam = 1.0 - np.exp(-ep/500.0)
    eq_v, co_v = float(sf['eq'][ki].mean()), float(sf['co'][ki].mean())
    viewerE.text_overlay.text = (f'{fr*100:g}% sampling   epoch {ep}   '
                                 f'λ_phys(t) = {lam:.2f}   '
                                 f'mean L_eq = {eq_v:.2e}   mean L_co = {co_v:.2e}')
viewerE.dims.events.current_step.connect(_labelE); _labelE()

---
## Curvenote / Web-Compatible Interactive Widgets  (ipywidgets + plotly)

The five napari views above require a desktop Qt display. The cells below are **headless-safe** alternatives — identical functionality rendered with **plotly** (3D surfaces, heatmaps) and **ipywidgets** sliders/play buttons. They work in:

- [Curvenote](https://curvenote.com) articles and notebooks
- JupyterHub / Binder
- VS Code Jupyter
- Classic Jupyter (with `jupyterlab-plotly` or `notebook>=7`)

**Prerequisites:** run cells 2–5 (config → data → training → PCA stats). The napari view cells (A–E above) do **not** need to run.

| Widget | Controls | What it shows |
|---|---|---|
| **A** | Sampling %, Statistic | Stacked translucent planes: input → sine layers → output |
| **B** | Sampling %, Field, Show | 3D strain surface — ground truth / PINN / \|error\| |
| **C** | Sampling %, Neuron ▶ | First-layer plane-wave gratings, variance-sorted |
| **D** | Sampling %, Field, Snapshot ▶ | Field emerging during training (Play for animation) |
| **E** | Sampling %, Snapshot ▶ | Layer stack animated across training + physics residuals |

In [10]:
# ---- widget setup: imports + pre-compute all arrays needed by widgets ------
# Self-contained. Depends only on cells 2–5 (no napari cells required).
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# ── helper functions (mirror of View A napari cell) ───────────────────────────
def _norm_sym(a, p=99.5):
    return np.clip(a / (np.percentile(np.abs(a), p) + 1e-12), -1, 1)

def _norm_minmax(a):
    return (a - a.min()) / (a.max() - a.min() + 1e-12) * 2 - 1

def _fixed_range_stack(arr, sym=False, logscale=False, p=99.5):
    """(n_frac, n_snap, Hd, Wd) → fixed per-fraction colour range across epochs."""
    out = np.empty_like(arr, dtype=np.float32)
    for i in range(arr.shape[0]):
        a = np.log10(arr[i] + 1e-8) if logscale else arr[i]
        if sym:
            lim = np.percentile(np.abs(a), p) + 1e-12
            out[i] = np.clip(a / lim, -1, 1)
        else:
            lo, hi = np.percentile(a, 100 - p), np.percentile(a, p)
            out[i] = np.clip((a - lo) / (hi - lo + 1e-12), 0, 1)
    return out

# ── downsampled grids ─────────────────────────────────────────────────────────
_gt_ds_w   = gt_maps[:, ::DS, ::DS]                               # (4, Hd, Wd)
_pred_ds_w = np.stack([preds[f][:, ::DS, ::DS] for f in FRACS])  # (n_f, 4, Hd, Wd)
_yg, _xg   = np.mgrid[0:Hd, 0:Wd].astype(np.float32) * DS

# ── dropdown option lists ─────────────────────────────────────────────────────
_frac_opts_w  = [(f'{f*100:g}%', i) for i, f in enumerate(FRACS)]
_field_opts_w = [(lbl, i) for i, lbl in enumerate(FIELD_LABELS)]

# ── first-layer sine gratings (View C) ────────────────────────────────────────
_grat_list, _orders_w, _w1s_w = [], [], []
for _fr in FRACS:
    with torch.no_grad():
        _a1 = torch.sin(models[_fr].first.w0 * models[_fr].first.linear(coords)).cpu().numpy()
    _o = np.argsort(-_a1.var(0))
    _grat_list.append(_a1.T[_o].reshape(-1, H, W)[:, ::DS, ::DS])
    _orders_w.append(_o)
    _w1s_w.append(models[_fr].first.linear.weight.detach().cpu().numpy())
_gratings_w = np.stack(_grat_list).astype(np.float32)  # (n_f, 128, Hd, Wd)

# ── layer activity PCA stats (reuse stats_by_frac from cell 5) ───────────────
_n_layers_w   = len(stats_by_frac[FRACS[0]])
_layer_nms_w  = ['sine layer 1 (input)'] + [f'sine layer {i+2}' for i in range(_n_layers_w - 1)]

# ── View-E stacks ─────────────────────────────────────────────────────────────
_gt_bcast_w  = np.broadcast_to(_gt_ds_w[0][None, None], (len(FRACS), N_SNAP, Hd, Wd))
_lkey        = 'layers' if 'layers' in snaps_by_frac[FRACS[0]] else 'units'
_layers_raw_w = np.stack([snaps_by_frac[fr][_lkey] for fr in FRACS])
if _layers_raw_w.ndim == 6:
    _layers_raw_w = np.abs(_layers_raw_w).mean(axis=3)
_out_raw_w   = np.stack([snaps_by_frac[fr]['pred'][:, 0] for fr in FRACS])
_eq_raw_w    = np.stack([snaps_by_frac[fr]['eq']  for fr in FRACS])
_co_raw_w    = np.stack([snaps_by_frac[fr]['co']  for fr in FRACS])

_meas_stk    = _fixed_range_stack(_gt_bcast_w, sym=True)
_lyr_stks_w  = [_fixed_range_stack(_layers_raw_w[:, :, l], sym=False)
                for l in range(_layers_raw_w.shape[2])]
_out_stk     = _fixed_range_stack(_out_raw_w, sym=True)
_eq_stk      = _fixed_range_stack(_eq_raw_w,  sym=False, logscale=True)
_co_stk      = _fixed_range_stack(_co_raw_w,  sym=False, logscale=True)

print(f'Widget setup done  —  {len(FRACS)} fractions · {N_SNAP} snapshots · '
      f'{_n_layers_w} sine layers · {_gratings_w.shape[1]} neurons')

Widget setup done  —  6 fractions · 26 snapshots · 5 sine layers · 128 neurons


In [ ]:
# ---- Widget A: layer stack ------------------------------------------------
# Stacked translucent surface planes (napari View A equivalent).
# Dropdowns: sampling fraction and statistic (PCA-1/2/3 or mean |activation|).
def widget_A():
    fw = widgets.Dropdown(options=_frac_opts_w, value=F0,
                          description='Sampling:', style={'description_width': 'initial'})
    sw = widgets.Dropdown(options=list(enumerate(STAT_KEYS)), value=0,
                          description='Statistic:', style={'description_width': 'initial'})

    plane_names = ['measured ε_xx'] + _layer_nms_w + ['reconstructed ε_xx']
    spacing = 2.0
    zz_flat = [np.full((Hd, Wd), k * spacing, dtype=np.float32) for k in range(len(plane_names))]

    def _arrays(fi, si):
        sk = STAT_KEYS[si]
        out = [_norm_sym(_gt_ds_w[0])]
        for l in range(_n_layers_w):
            a = stats_by_frac[FRACS[fi]][l][sk]
            out.append(_norm_minmax(a) if sk == 'mean |activation|' else _norm_sym(a))
        out.append(_norm_sym(_pred_ds_w[fi, 0]))
        return out

    fig = go.FigureWidget()
    arrs0 = _arrays(F0, 0)
    for k, (nm, zz, arr) in enumerate(zip(plane_names, zz_flat, arrs0)):
        fig.add_trace(go.Surface(
            x=_xg, y=_yg, z=zz, surfacecolor=arr,
            colorscale='RdBu', cmin=-1, cmax=1,
            showscale=(k == 0), opacity=0.85, name=nm,
            hovertemplate=f'<b>{nm}</b><br>val=%{{surfacecolor:.3f}}<extra></extra>'
        ))
    fig.update_layout(
        title=f'A — Layer stack  |  {FRACS[F0]*100:g}%  |  {STAT_KEYS[0]}',
        scene=dict(xaxis_title='x (px)', yaxis_title='y (px)', zaxis_title='layer depth',
                   camera=dict(eye=dict(x=1.5, y=-1.5, z=1.2))),
        height=650, margin=dict(l=0, r=0, b=0, t=40)
    )

    def _update(change):
        fi, si = fw.value, sw.value
        arrs = _arrays(fi, si)
        with fig.batch_update():
            for k, arr in enumerate(arrs):
                fig.data[k].surfacecolor = arr
            fig.layout.title.text = f'A — Layer stack  |  {FRACS[fi]*100:g}%  |  {STAT_KEYS[si]}'

    fw.observe(_update, 'value'); sw.observe(_update, 'value')
    display(widgets.VBox([widgets.HBox([fw, sw]), fig]))

widget_A()

TraitError: Invalid selection: value not found

: 

In [ ]:
# ---- Widget B: field explorer 3D surface ----------------------------------
# Height = scaled field value; colour = field value.
# Dropdowns: fraction, field component, layer (GT / PINN / |error|).
def widget_B():
    fw = widgets.Dropdown(options=_frac_opts_w, value=F0,
                          description='Sampling:', style={'description_width': 'initial'})
    cw = widgets.Dropdown(options=_field_opts_w, value=0,
                          description='Field:', style={'description_width': 'initial'})
    lw = widgets.ToggleButtons(
        options=['Ground truth', 'PINN', '|Error|'], value='PINN',
        description='Show:', style={'description_width': 'initial',
                                    'button_width': '120px'}
    )
    Z_S = 60.0

    def _surf(fi, ci, layer):
        gt_c = _gt_ds_w[ci].astype(float)
        pr_c = _pred_ds_w[fi, ci].astype(float)
        field = gt_c if layer == 'Ground truth' else (pr_c if layer == 'PINN' else np.abs(pr_c - gt_c))
        lo, hi = field.min(), field.max()
        z = (field - lo) / (hi - lo + 1e-12) * Z_S
        vm = max(abs(gt_c.min()), abs(gt_c.max()), abs(pr_c.min()), abs(pr_c.max()))
        return z, field, vm

    z0, f0, vm0 = _surf(F0, 0, 'PINN')
    fig = go.FigureWidget()
    fig.add_trace(go.Surface(
        x=_xg, y=_yg, z=z0, surfacecolor=f0,
        colorscale='RdBu', cmin=-vm0, cmax=vm0, showscale=True,
        hovertemplate='x=%{x:.0f}  y=%{y:.0f}  val=%{surfacecolor:.4f}<extra></extra>'
    ))
    fig.update_layout(
        title=f'B — Field explorer  |  {FIELD_LABELS[0]}  |  PINN  |  {FRACS[F0]*100:g}%',
        scene=dict(xaxis_title='x (px)', yaxis_title='y (px)', zaxis_title='scaled field',
                   camera=dict(eye=dict(x=1.5, y=-1.5, z=1.2))),
        height=600, margin=dict(l=0, r=0, b=0, t=40)
    )

    def _update(change):
        fi, ci, layer = fw.value, cw.value, lw.value
        z, field, vm = _surf(fi, ci, layer)
        with fig.batch_update():
            fig.data[0].z = z
            fig.data[0].surfacecolor = field
            if layer == '|Error|':
                fig.data[0].colorscale = 'Hot'
                fig.data[0].cmin = 0
                fig.data[0].cmax = float(field.max()) + 1e-12
            else:
                fig.data[0].colorscale = 'RdBu'
                fig.data[0].cmin = -vm
                fig.data[0].cmax = vm
            fig.layout.title.text = (
                f'B — Field explorer  |  {FIELD_LABELS[ci]}  |  {layer}  |  {FRACS[fi]*100:g}%'
            )

    fw.observe(_update, 'value'); cw.observe(_update, 'value'); lw.observe(_update, 'value')
    display(widgets.VBox([widgets.HBox([fw, cw]), lw, fig]))

widget_B()

In [ ]:
# ---- Widget C: first-layer sine gratings ----------------------------------
# Heatmap of sin(ω₀(w·x)+b) for each neuron.
# Press ▶ on the Play button to cycle through all 128 neurons.
def widget_C():
    n_neurons = _gratings_w.shape[1]
    fw   = widgets.Dropdown(options=_frac_opts_w, value=F0,
                            description='Sampling:', style={'description_width': 'initial'})
    nw   = widgets.IntSlider(min=0, max=n_neurons - 1, value=0, continuous_update=False,
                             description='Neuron rank:', style={'description_width': 'initial'})
    play = widgets.Play(min=0, max=n_neurons - 1, step=1, interval=120, description='▶')
    widgets.jslink((play, 'value'), (nw, 'value'))

    fig = go.FigureWidget()
    fig.add_trace(go.Heatmap(
        z=_gratings_w[F0, 0], colorscale='RdBu', zmin=-1, zmax=1,
        hovertemplate='x=%{x}  y=%{y}  val=%{z:.3f}<extra></extra>'
    ))
    fig.update_layout(
        title=f'C — Sine gratings  |  {FRACS[F0]*100:g}%  |  neuron 0 (rank 1)',
        xaxis=dict(title='x (px)', scaleanchor='y', constrain='domain'),
        yaxis_title='y (px)',
        height=480, margin=dict(l=0, r=0, b=0, t=40)
    )

    def _update(change):
        fi, ri = fw.value, nw.value
        n  = int(_orders_w[fi][ri])
        fx, fy = OMEGA_0 * _w1s_w[fi][n] / (2 * np.pi)
        with fig.batch_update():
            fig.data[0].z = _gratings_w[fi, ri]
            fig.layout.title.text = (
                f'C — Sine gratings  |  {FRACS[fi]*100:g}%  |  '
                f'neuron {n} (rank {ri + 1})  |  '
                f'freq ≈ ({fx:+.1f}, {fy:+.1f}) cycles/unit'
            )

    fw.observe(_update, 'value'); nw.observe(_update, 'value')
    display(widgets.VBox([widgets.HBox([fw, play, nw]), fig]))

widget_C()

In [ ]:
# ---- Widget D: training evolution ----------------------------------------
# Heatmap of the PINN prediction at each snapshot epoch.
# Press ▶ on the Play button to watch the field emerge during training.
def widget_D():
    fw   = widgets.Dropdown(options=_frac_opts_w, value=F0,
                            description='Sampling:', style={'description_width': 'initial'})
    cw   = widgets.Dropdown(options=_field_opts_w, value=0,
                            description='Field:', style={'description_width': 'initial'})
    kw   = widgets.IntSlider(min=0, max=N_SNAP - 1, value=0, continuous_update=False,
                             description='Snapshot:', style={'description_width': 'initial'})
    play = widgets.Play(min=0, max=N_SNAP - 1, step=1, interval=300, description='▶')
    gt_w = widgets.Checkbox(value=False, description='Show ground truth')
    widgets.jslink((play, 'value'), (kw, 'value'))

    def _frame(fi, ci, ki):
        fr = FRACS[fi]; sf = snaps_by_frac[fr]
        sc = np.percentile(np.abs(_gt_ds_w[ci]), 99.5) + 1e-12
        return (sf['pred'][ki, ci] / sc,
                _gt_ds_w[ci] / sc,
                int(sf['epoch'][ki]),
                float(sf['val'][ki]))

    pred0, gt0, ep0, vl0 = _frame(F0, 0, 0)
    fig = go.FigureWidget()
    fig.add_trace(go.Heatmap(z=pred0, colorscale='RdBu', zmin=-1, zmax=1,
                              name='PINN'))
    fig.add_trace(go.Heatmap(z=gt0,   colorscale='RdBu', zmin=-1, zmax=1,
                              name='Ground truth', visible=False))
    fig.update_layout(
        title=(f'D — Training evolution  |  {FRACS[F0]*100:g}%  |  {FIELD_LABELS[0]}  |  '
               f'epoch {ep0}  |  val = {vl0:.4f}'),
        xaxis_title='x (px)', yaxis_title='y (px)',
        height=490, margin=dict(l=0, r=0, b=0, t=40),
        legend=dict(orientation='h', y=-0.15)
    )

    def _update(change):
        fi, ci, ki = fw.value, cw.value, kw.value
        pred, gt_f, ep, vl = _frame(fi, ci, ki)
        with fig.batch_update():
            fig.data[0].z = pred
            fig.data[1].z = gt_f
            fig.data[1].visible = gt_w.value
            fig.layout.title.text = (
                f'D — Training evolution  |  {FRACS[fi]*100:g}%  |  {FIELD_LABELS[ci]}  |  '
                f'epoch {ep}  |  val = {vl:.4f}'
            )

    for w in (fw, cw, kw, gt_w):
        w.observe(_update, 'value')
    display(widgets.VBox([
        widgets.HBox([fw, cw, gt_w]),
        widgets.HBox([play, kw]),
        fig
    ]))

widget_D()

In [ ]:
# ---- Widget E: layer stack, training evolution ----------------------------
# Stacked translucent planes (input → layers → output → residuals) animated
# across training snapshots. Press ▶ to watch the network learn the field
# while the physics-residual planes darken. Text overlay tracks λ_phys(t).
def widget_E():
    fw   = widgets.Dropdown(options=_frac_opts_w, value=F0,
                            description='Sampling:', style={'description_width': 'initial'})
    kw   = widgets.IntSlider(min=0, max=N_SNAP - 1, value=0, continuous_update=False,
                             description='Snapshot:', style={'description_width': 'initial'})
    play = widgets.Play(min=0, max=N_SNAP - 1, step=1, interval=400, description='▶')
    widgets.jslink((play, 'value'), (kw, 'value'))

    plane_names = (['measured ε_xx'] + _layer_nms_w +
                   ['reconstructed ε_xx', 'equilibrium residual (log)', 'compat. residual (log)'])
    spacing = 2.0
    zz_e = [np.full((Hd, Wd), k * spacing, dtype=np.float32) for k in range(len(plane_names))]
    # colormaps: RdBu for symmetric fields, Viridis for layer activity, Hot for residuals
    cmaps = (['RdBu'] +
             ['Viridis'] * _n_layers_w +
             ['RdBu', 'Hot', 'Hot'])
    clims = ([(-1, 1)] +
             [(0, 1)] * _n_layers_w +
             [(-1, 1), (0, 1), (0, 1)])

    def _arrays(fi, ki):
        out = [_meas_stk[fi, ki]]
        for l in range(_n_layers_w):
            out.append(_lyr_stks_w[l][fi, ki])
        out.extend([_out_stk[fi, ki], _eq_stk[fi, ki], _co_stk[fi, ki]])
        return out

    fig = go.FigureWidget()
    arrs0 = _arrays(F0, 0)
    for k, (nm, zz, arr, cmap, clim) in enumerate(zip(plane_names, zz_e, arrs0, cmaps, clims)):
        fig.add_trace(go.Surface(
            x=_xg, y=_yg, z=zz, surfacecolor=arr,
            colorscale=cmap, cmin=clim[0], cmax=clim[1],
            showscale=(k == 0), opacity=0.85, name=nm,
            hovertemplate=f'<b>{nm}</b><br>val=%{{surfacecolor:.3f}}<extra></extra>'
        ))
    _sf0_e = snaps_by_frac[FRACS[F0]]
    _ep0_e = int(_sf0_e['epoch'][0])
    _lam0  = 1.0 - np.exp(-_ep0_e / 500.0)
    fig.update_layout(
        title=(f'E — Layer stack, training evolution  |  {FRACS[F0]*100:g}%  |  '
               f'epoch {_ep0_e}  |  λ = {_lam0:.2f}'),
        scene=dict(xaxis_title='x (px)', yaxis_title='y (px)', zaxis_title='layer depth',
                   camera=dict(eye=dict(x=1.5, y=-1.5, z=1.2))),
        height=680, margin=dict(l=0, r=0, b=0, t=40)
    )

    def _update(change):
        fi, ki = fw.value, kw.value
        fr = FRACS[fi]; sf = snaps_by_frac[fr]
        ep  = int(sf['epoch'][ki])
        lam = 1.0 - np.exp(-ep / 500.0)
        eq_v = float(sf['eq'][ki].mean()); co_v = float(sf['co'][ki].mean())
        arrs = _arrays(fi, ki)
        with fig.batch_update():
            for k, arr in enumerate(arrs):
                fig.data[k].surfacecolor = arr
            fig.layout.title.text = (
                f'E — Layer stack, training evolution  |  {fr*100:g}%  |  '
                f'epoch {ep}  |  λ = {lam:.2f}  |  '
                f'L_eq = {eq_v:.2e}  L_co = {co_v:.2e}'
            )

    fw.observe(_update, 'value'); kw.observe(_update, 'value')
    display(widgets.VBox([widgets.HBox([fw, play, kw]), fig]))

widget_E()